# Module 23 — Entity / key-value memory

**THE ONE IDEA:** the cheapest, most reliable memory tier is **a dict**. Most people skip
straight from "stuff it in the prompt" to "embed it in a vector DB" and never build this
one.

Module 22 ended with a fact that must never be lost — a client reference. A summariser
*might* keep it. A vector search *might* retrieve it. **A key-value lookup always
returns it**, in constant time, with no model in the path.

```
semantic memory = de-tensed FACTS about entities
                  (user_mx7741, product_interest, "5-year fix")
```

Use this tier when the fact is **structured, exact, and must not be approximate.**


In [ ]:
import sys; sys.path.insert(0, "..")   # shared _providers.py / _tools.py at the phase root
from _providers import get_client
from pydantic import BaseModel
from typing import Literal

client, MODEL, _ = get_client("openai")

STORE: dict[str, dict] = {}          # entity -> {attribute: {value, confidence, source}}

def write_fact(entity, attr, value, confidence, source):
    STORE.setdefault(entity, {})[attr] = {"value": value, "confidence": confidence,
                                          "source": source}

def read_entity(entity): return STORE.get(entity, {})

class Fact(BaseModel):
    entity: str
    attribute: Literal["reference", "product_interest", "employment", "ltv_target"]
    value: str
    confidence: float

## Extraction — decide what is worth storing

Not everything said is worth remembering. A confidence threshold and a closed set of
attributes keep the store clean. **Writing too much pollutes retrieval**, which is the
failure people hit after a month in production.

In [ ]:
def extract(utterance):
    s = Fact.model_json_schema(); s["additionalProperties"] = False
    r = client.chat.completions.create(model=MODEL, max_tokens=200,
        response_format={"type": "json_schema",
                         "json_schema": {"name": "fact", "strict": True, "schema": s}},
        messages=[{"role": "user", "content":
                   "Extract ONE durable fact about the client. confidence 0-1.\n\n" + utterance}])
    return Fact.model_validate_json(r.choices[0].message.content)

UTTERANCES = ["My client reference is MX-7741 and she wants a 5-year fix.",
              "She's self-employed, two years of accounts.",
              "Ok thanks, that's helpful.",                      # <- nothing durable here
              "She's targeting 90% LTV."]

for u in UTTERANCES:
    f = extract(u)
    keep = f.confidence >= 0.7
    print(f"  {f.attribute:17} = {f.value[:26]:26} conf={f.confidence:.2f}  "
          f"{'STORED' if keep else 'skipped'}")
    if keep:
        write_fact("client:MX-7741", f.attribute, f.value, f.confidence, u[:34])

## Read it back — no model, no similarity, no guessing

In [ ]:
for attr, rec in read_entity("client:MX-7741").items():
    print(f"  {attr:17} {rec['value'][:30]:30} conf={rec['confidence']:.2f}  "
          f"src={rec['source'][:26]!r}")

## Conflict resolution

Facts change. "Latest wins" is the right default for a preference; **source priority**
is right for ground truth from a system of record.

In [ ]:
def write_with_policy(entity, attr, value, confidence, source, policy="latest"):
    cur = STORE.get(entity, {}).get(attr)
    if cur and policy == "source_priority" and cur["source"].startswith("SYSTEM"):
        print(f"  KEPT {attr}={cur['value']!r} (system-of-record beats a chat claim)")
        return
    if cur:
        print(f"  CONFLICT {attr}: {cur['value']!r} -> {value!r}  [{policy}]")
    write_fact(entity, attr, value, confidence, source)

write_fact("client:MX-7741", "employment", "self-employed", 0.95, "SYSTEM:hr-feed")
write_with_policy("client:MX-7741", "ltv_target", "85%", 0.9, "turn 9", "latest")
write_with_policy("client:MX-7741", "employment", "employed", 0.6, "turn 10", "source_priority")

print("\nfinal state:")
for a, r in read_entity("client:MX-7741").items():
    print(f"  {a:17} {r['value']}")

print("""
LESSON - this tier is a dict, and it beats a vector store on every axis that
matters for EXACT facts:

  exact        'MX-7741' comes back as 'MX-7741', never as something similar
  cheap        no embedding call, no index, no similarity threshold to tune
  auditable    every fact carries value + confidence + SOURCE. You can answer
               'why does the agent think she is self-employed?' - module 14's
               requirement, applied to memory
  correctable  one assignment updates it. Deleting from a vector index is work

What it CANNOT do is answer 'what did we discuss about overpayments?', because
that is a similarity question over unstructured text. THAT is what module 24 is
for.

The production mistake is reaching for 24 first. Most of what an agent needs to
remember about a user is a dozen structured fields, and every one of them is
cheaper, faster and more trustworthy here.""")

---

**Next:** `24_memory_long_term_vector.ipynb`